# Simulate Provider Source Systems

This notebook creates three source-system datasets from the provider ground truth:

- EHR
- HR
- Credentialing

The records are based on known providers, but each source uses a different schema and contains controlled differences such as missing values, name variations, phone formatting, address abbreviations, and a small number of typographical errors.

The source files do not keep `ground_truth_id`. A separate validation map records the known relationship between each simulated source record and the original provider. That map should only be used when evaluating matching results later in the project.

## 1. Setup

Define the input and output locations and keep the simulation settings in one place. A fixed random seed makes the generated datasets reproducible.

In [8]:
from pathlib import Path
import re

import numpy as np
import pandas as pd


GROUND_TRUTH_FILE = Path("../data/ground_truth/provider_ground_truth.csv")

SIMULATED_DIR = Path("../data/simulated")
VALIDATION_DIR = Path("../data/validation")

SIMULATED_DIR.mkdir(parents=True, exist_ok=True)
VALIDATION_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)

# Source coverage
EHR_COVERAGE = 1.00
HR_COVERAGE = 0.75
CREDENTIALING_COVERAGE = 0.85


## 2. Load the ground truth

Load identifiers as strings so values such as NPI, ZIP code, phone number, and license number are not converted to numeric types.

In [9]:
ground_truth = pd.read_csv(
    GROUND_TRUTH_FILE,
    dtype="string"
)

print(f"Ground-truth providers: {len(ground_truth):,}")
ground_truth.head()


Ground-truth providers: 3,000


,ground_truth_id,npi,entity_type_code,first_name,middle_name,last_name,name_prefix,name_suffix,credential,practice_address_line_1,...,practice_city,practice_state,practice_zip,practice_phone,primary_taxonomy_code,primary_license_number,primary_license_state,enumeration_date,last_update_date,npi_status
0,GT00001,1154066645,1,SEAN,THOMAS,HEALY,DR.,<NA>,DO,27700 NORTHWEST FWY STE 200,...,CYPRESS,TX,774336767,7134865750,207QS0010X,V9012,TX,04/29/2022,08/05/2026,ACTIVE
1,GT00002,1699411744,1,ARIEL,LEI,BURRIS,<NA>,<NA>,"BA, RBT",6455 S SHORE BLVD STE 400,...,LEAGUE CITY,TX,775735525,8329329344,106S00000X,<NA>,<NA>,05/11/2022,05/11/2022,ACTIVE
2,GT00003,1669205571,1,RHONDA,<NA>,SAMUEL,<NA>,<NA>,<NA>,9815 MERIDIANA PKWY,...,ARCOLA,TX,775833582,2812453878,235Z00000X,121434,TX,08/26/2024,08/26/2024,ACTIVE
3,GT00004,1265081889,1,MATTHEW,COLE,RAMOS,DR.,<NA>,DC,5152 69TH ST STE 101,...,LUBBOCK,TX,794241661,8067944009,111N00000X,14134,TX,09/06/2019,09/06/2019,ACTIVE
4,GT00005,1609044783,1,DOMONIQUE,<NA>,RANDALL,<NA>,<NA>,<NA>,26006 OAKRIDGE DR.,...,THE WOODLANDS,TX,77380,8323582655,174400000X,1-00-0350,TX,02/12/2008,02/12/2008,ACTIVE


## 3. Validate the expected schema

The simulation depends on the fields created in the ground-truth notebook. Stop early if any required field is missing rather than generating incomplete source files.

In [10]:
required_columns = [
    "ground_truth_id",
    "npi",
    "first_name",
    "middle_name",
    "last_name",
    "credential",
    "practice_address_line_1",
    "practice_address_line_2",
    "practice_city",
    "practice_state",
    "practice_zip",
    "practice_phone",
    "primary_taxonomy_code",
    "primary_license_number",
    "primary_license_state",
]

missing_columns = [
    column
    for column in required_columns
    if column not in ground_truth.columns
]

if missing_columns:
    raise ValueError(
        f"Ground-truth file is missing required columns: {missing_columns}"
    )

assert ground_truth["ground_truth_id"].is_unique
assert ground_truth["npi"].is_unique
assert ground_truth["npi"].notna().all()

print("Ground-truth schema checks passed.")


Ground-truth schema checks passed.


## 4. Define source coverage

Every provider is included in the EHR dataset. HR and Credentialing contain overlapping subsets of the same provider population.

This creates a more realistic source landscape while ensuring every ground-truth provider appears in at least one operational system.

In [11]:
ehr_base = ground_truth.copy()

hr_base = (
    ground_truth
    .sample(
        frac=HR_COVERAGE,
        random_state=RANDOM_SEED + 1
    )
    .reset_index(drop=True)
)

credentialing_base = (
    ground_truth
    .sample(
        frac=CREDENTIALING_COVERAGE,
        random_state=RANDOM_SEED + 2
    )
    .reset_index(drop=True)
)

print(f"EHR providers:           {len(ehr_base):,}")
print(f"HR providers:            {len(hr_base):,}")
print(f"Credentialing providers: {len(credentialing_base):,}")


EHR providers:           3,000
HR providers:            2,250
Credentialing providers: 2,550


## 5. Add source-system identifiers

Each system uses its own record identifier. These IDs identify records inside the source system and are separate from the provider's NPI or internal ground-truth ID.

In [12]:
ehr_base.insert(
    0,
    "ehr_provider_id",
    [f"EHR{i:06d}" for i in range(1, len(ehr_base) + 1)]
)

hr_base.insert(
    0,
    "employee_id",
    [f"EMP{i:06d}" for i in range(1, len(hr_base) + 1)]
)

credentialing_base.insert(
    0,
    "credentialing_id",
    [f"CRED{i:06d}" for i in range(1, len(credentialing_base) + 1)]
)


## 6. Simulation helpers

These functions introduce controlled source-system differences. The goal is not to make the data unusable; it is to create realistic inconsistencies that the cleaning and matching steps will need to handle.

In [13]:
def random_mask(size, rate):
    """Return a reproducible random mask for a requested change rate."""
    return rng.random(size) < rate


def set_missing(df, column, rate):
    """Replace a percentage of non-null values with missing values."""
    mask = random_mask(len(df), rate) & df[column].notna()
    df.loc[mask, column] = pd.NA


def use_middle_initial(df, column, rate):
    """Replace some full middle names with a one-character initial."""
    mask = random_mask(len(df), rate) & df[column].notna()
    df.loc[mask, column] = (
        df.loc[mask, column]
        .str.strip()
        .str[:1]
    )


def format_phone_value(value, style):
    """Apply a source-specific display format to a 10-digit phone number."""
    if pd.isna(value):
        return pd.NA

    digits = re.sub(r"\D", "", str(value))

    if len(digits) != 10:
        return value

    area = digits[:3]
    prefix = digits[3:6]
    line = digits[6:]

    if style == "parentheses":
        return f"({area}) {prefix}-{line}"

    if style == "hyphen":
        return f"{area}-{prefix}-{line}"

    if style == "plain":
        return digits

    return value


def format_phone_column(df, column, style, rate):
    """Apply phone formatting to a percentage of populated values."""
    mask = random_mask(len(df), rate) & df[column].notna()
    df.loc[mask, column] = df.loc[mask, column].map(
        lambda value: format_phone_value(value, style)
    )


def abbreviate_address(df, column, rate):
    """Apply common street-type abbreviations to selected addresses."""
    mask = random_mask(len(df), rate) & df[column].notna()

    replacements = {
        r"\bSTREET\b": "ST",
        r"\bROAD\b": "RD",
        r"\bAVENUE\b": "AVE",
        r"\bBOULEVARD\b": "BLVD",
        r"\bDRIVE\b": "DR",
        r"\bLANE\b": "LN",
        r"\bCOURT\b": "CT",
        r"\bPARKWAY\b": "PKWY",
        r"\bHIGHWAY\b": "HWY",
        r"\bSUITE\b": "STE",
    }

    values = df.loc[mask, column].copy()

    for pattern, replacement in replacements.items():
        values = values.str.replace(
            pattern,
            replacement,
            regex=True,
            case=False
        )

    df.loc[mask, column] = values


def change_case(df, column, rate, style):
    """Change text casing for selected source values."""
    mask = random_mask(len(df), rate) & df[column].notna()

    if style == "upper":
        df.loc[mask, column] = df.loc[mask, column].str.upper()

    elif style == "lower":
        df.loc[mask, column] = df.loc[mask, column].str.lower()

    elif style == "title":
        df.loc[mask, column] = df.loc[mask, column].str.title()


def drop_one_character(value):
    """Create a small typographical error by removing one internal character."""
    if pd.isna(value):
        return pd.NA

    text = str(value)

    if len(text) < 5:
        return text

    position = int(rng.integers(1, len(text) - 1))
    return text[:position] + text[position + 1:]


def introduce_typo(df, column, rate):
    """Apply a small character-drop typo to a limited number of values."""
    mask = random_mask(len(df), rate) & df[column].notna()
    df.loc[mask, column] = df.loc[mask, column].map(drop_one_character)


## 7. Build the EHR source

The EHR keeps patient-care-facing provider information such as NPI, name, specialty code, practice location, and phone number.

The column names are changed to reflect the EHR schema rather than copying the ground-truth table directly.

In [14]:
ehr = ehr_base[
    [
        "ground_truth_id",
        "ehr_provider_id",
        "npi",
        "first_name",
        "middle_name",
        "last_name",
        "credential",
        "primary_taxonomy_code",
        "practice_address_line_1",
        "practice_address_line_2",
        "practice_city",
        "practice_state",
        "practice_zip",
        "practice_phone",
    ]
].copy()

ehr = ehr.rename(columns={
    "primary_taxonomy_code": "specialty_code",
    "practice_address_line_1": "address_line_1",
    "practice_address_line_2": "address_line_2",
    "practice_city": "city",
    "practice_state": "state",
    "practice_zip": "zip",
    "practice_phone": "phone",
})

ehr.head()


,ground_truth_id,ehr_provider_id,npi,first_name,middle_name,last_name,credential,specialty_code,address_line_1,address_line_2,city,state,zip,phone
0,GT00001,EHR000001,1154066645,SEAN,THOMAS,HEALY,DO,207QS0010X,27700 NORTHWEST FWY STE 200,<NA>,CYPRESS,TX,774336767,7134865750
1,GT00002,EHR000002,1699411744,ARIEL,LEI,BURRIS,"BA, RBT",106S00000X,6455 S SHORE BLVD STE 400,<NA>,LEAGUE CITY,TX,775735525,8329329344
2,GT00003,EHR000003,1669205571,RHONDA,<NA>,SAMUEL,<NA>,235Z00000X,9815 MERIDIANA PKWY,<NA>,ARCOLA,TX,775833582,2812453878
3,GT00004,EHR000004,1265081889,MATTHEW,COLE,RAMOS,DC,111N00000X,5152 69TH ST STE 101,<NA>,LUBBOCK,TX,794241661,8067944009
4,GT00005,EHR000005,1609044783,DOMONIQUE,<NA>,RANDALL,<NA>,174400000X,26006 OAKRIDGE DR.,<NA>,THE WOODLANDS,TX,77380,8323582655


### Apply EHR variations

EHR records keep most NPIs, but some are missing. Middle names are often shortened, addresses and phone numbers use display formatting, and a small percentage of names contain typographical errors.

In [15]:
use_middle_initial(ehr, "middle_name", rate=0.60)
set_missing(ehr, "middle_name", rate=0.10)

set_missing(ehr, "npi", rate=0.08)
set_missing(ehr, "phone", rate=0.05)
set_missing(ehr, "address_line_2", rate=0.20)

format_phone_column(
    ehr,
    "phone",
    style="parentheses",
    rate=0.70
)

abbreviate_address(
    ehr,
    "address_line_1",
    rate=0.65
)

change_case(ehr, "first_name", rate=0.25, style="title")
change_case(ehr, "last_name", rate=0.25, style="title")

introduce_typo(ehr, "first_name", rate=0.01)
introduce_typo(ehr, "last_name", rate=0.01)


## 8. Build the HR source

HR focuses on employee and workforce information. It keeps fewer clinical fields than the EHR or Credentialing systems.

NPI is included when available in the HR record, but it is intentionally incomplete so that later matching cannot depend on NPI alone.

In [16]:
hr = hr_base[
    [
        "ground_truth_id",
        "employee_id",
        "npi",
        "first_name",
        "middle_name",
        "last_name",
        "credential",
        "primary_taxonomy_code",
        "practice_city",
        "practice_state",
        "practice_phone",
    ]
].copy()

hr = hr.rename(columns={
    "middle_name": "middle_initial",
    "credential": "job_credential",
    "primary_taxonomy_code": "job_specialty_code",
    "practice_city": "work_city",
    "practice_state": "work_state",
    "practice_phone": "work_phone",
})

hr.head()


,ground_truth_id,employee_id,npi,first_name,middle_initial,last_name,job_credential,job_specialty_code,work_city,work_state,work_phone
0,GT01324,EMP000001,1083933857,JOHANNA,A,RAMIREZ,MOTR,225X00000X,SAN ANTONIO,TX,8009449782
1,GT01388,EMP000002,1487266987,ESEER,<NA>,AL RUKABI,RPH,183500000X,AUSTIN,TX,5124598308
2,GT02337,EMP000003,1760690853,PAMELA,KAYE,HANCOCK,LPC,101YP2500X,DENTON,TX,9404847799
3,GT01858,EMP000004,1699706085,RYAN,CHRISTOPHER,BLISS,PT,225100000X,BASTROP,TX,5123031116
4,GT02508,EMP000005,1881252849,MORGAN,<NA>,STURGEON,<NA>,106S00000X,KILLEEN,TX,2545541466


### Apply HR variations

HR has the largest amount of missing NPI data in the simulation. Names are more likely to use middle initials, and phone formatting differs from the EHR.

In [17]:
use_middle_initial(hr, "middle_initial", rate=0.85)
set_missing(hr, "middle_initial", rate=0.15)

set_missing(hr, "npi", rate=0.55)
set_missing(hr, "work_phone", rate=0.12)
set_missing(hr, "job_specialty_code", rate=0.20)

format_phone_column(
    hr,
    "work_phone",
    style="hyphen",
    rate=0.80
)

change_case(hr, "first_name", rate=0.20, style="upper")
change_case(hr, "last_name", rate=0.35, style="upper")

introduce_typo(hr, "first_name", rate=0.02)
introduce_typo(hr, "last_name", rate=0.02)


## 9. Build the Credentialing source

Credentialing keeps the fields used to verify a provider's professional identity, including NPI, legal name, credential, taxonomy, license number, license state, and practice contact information.

This source is kept cleaner than HR because identity and licensing fields are central to the credentialing process.

In [18]:
credentialing = credentialing_base[
    [
        "ground_truth_id",
        "credentialing_id",
        "npi",
        "first_name",
        "middle_name",
        "last_name",
        "credential",
        "primary_taxonomy_code",
        "primary_license_number",
        "primary_license_state",
        "practice_address_line_1",
        "practice_address_line_2",
        "practice_city",
        "practice_state",
        "practice_zip",
        "practice_phone",
    ]
].copy()

credentialing = credentialing.rename(columns={
    "first_name": "legal_first_name",
    "middle_name": "legal_middle_name",
    "last_name": "legal_last_name",
    "primary_taxonomy_code": "taxonomy_code",
    "primary_license_number": "license_number",
    "primary_license_state": "license_state",
    "practice_address_line_1": "address_line_1",
    "practice_address_line_2": "address_line_2",
    "practice_city": "city",
    "practice_state": "state",
    "practice_zip": "zip",
    "practice_phone": "phone",
})

credentialing.head()


,ground_truth_id,credentialing_id,npi,legal_first_name,legal_middle_name,legal_last_name,credential,taxonomy_code,license_number,license_state,address_line_1,address_line_2,city,state,zip,phone
0,GT01424,CRED000001,1205445301,KEEGAN,<NA>,MATTOX,PHARMD,1835P0018X,65602,TX,160 N COIT RD,<NA>,RICHARDSON,TX,750805454,9724979339
1,GT01276,CRED000002,1821717257,ALEXANDRA,MARIE,GARZA,<NA>,235Z00000X,<NA>,<NA>,1700 WILSON RD,<NA>,HUMBLE,TX,773386118,2816416357
2,GT01727,CRED000003,1558890962,THEODORA,EFROSENI,TSAKALAKIS,ATC,2255A2300X,2000027672,TX,15722 DUNMOOR DR.,<NA>,HOUSTON,TX,77059,2814802070
3,GT00897,CRED000004,1215255146,RYAN,SAMUEL,STEPINOFF,PA-C,363AM0700X,17487375,TX,6913 CAMP BOWIE BLVD STE 141,<NA>,FORT WORTH,TX,761167165,8175604540
4,GT01457,CRED000005,1396886198,JOHN,EMERSON,WINEMAN,PA-C,363A00000X,PA02729,TX,120 WOOD AVE,<NA>,WOODSBORO,TX,78393,3615435414


### Apply Credentialing variations

Credentialing retains most identity values but still contains normal operational inconsistencies such as missing contact information, address abbreviations, and occasional shortened middle names.

In [19]:
use_middle_initial(
    credentialing,
    "legal_middle_name",
    rate=0.20
)

set_missing(
    credentialing,
    "legal_middle_name",
    rate=0.05
)

set_missing(credentialing, "npi", rate=0.02)
set_missing(credentialing, "phone", rate=0.08)

format_phone_column(
    credentialing,
    "phone",
    style="plain",
    rate=0.80
)

abbreviate_address(
    credentialing,
    "address_line_1",
    rate=0.30
)

change_case(
    credentialing,
    "legal_first_name",
    rate=0.15,
    style="upper"
)

change_case(
    credentialing,
    "legal_last_name",
    rate=0.15,
    style="upper"
)

introduce_typo(
    credentialing,
    "legal_last_name",
    rate=0.005
)


## 10. Build the hidden validation map

The validation map stores the known relationship between each simulated record and its original provider.

This file is not part of the matching input. It is only used after matching to measure whether records were linked to the correct provider.

In [20]:
ehr_truth = ehr[
    [
        "ground_truth_id",
        "ehr_provider_id",
    ]
].copy()

ehr_truth["source_system"] = "EHR"
ehr_truth = ehr_truth.rename(
    columns={"ehr_provider_id": "source_record_id"}
)


hr_truth = hr[
    [
        "ground_truth_id",
        "employee_id",
    ]
].copy()

hr_truth["source_system"] = "HR"
hr_truth = hr_truth.rename(
    columns={"employee_id": "source_record_id"}
)


credentialing_truth = credentialing[
    [
        "ground_truth_id",
        "credentialing_id",
    ]
].copy()

credentialing_truth["source_system"] = "CREDENTIALING"
credentialing_truth = credentialing_truth.rename(
    columns={"credentialing_id": "source_record_id"}
)


simulation_truth_map = pd.concat(
    [
        ehr_truth,
        hr_truth,
        credentialing_truth,
    ],
    ignore_index=True
)

simulation_truth_map = simulation_truth_map[
    [
        "source_system",
        "source_record_id",
        "ground_truth_id",
    ]
]

simulation_truth_map.head()


,source_system,source_record_id,ground_truth_id
0,EHR,EHR000001,GT00001
1,EHR,EHR000002,GT00002
2,EHR,EHR000003,GT00003
3,EHR,EHR000004,GT00004
4,EHR,EHR000005,GT00005


### Add the original NPI to the validation map

Keeping the original NPI in the hidden map makes final match review easier without exposing the value to source records where NPI was intentionally removed.

In [21]:
simulation_truth_map = simulation_truth_map.merge(
    ground_truth[
        [
            "ground_truth_id",
            "npi",
        ]
    ].rename(columns={"npi": "truth_npi"}),
    on="ground_truth_id",
    how="left",
    validate="many_to_one"
)

simulation_truth_map.head()


,source_system,source_record_id,ground_truth_id,truth_npi
0,EHR,EHR000001,GT00001,1154066645
1,EHR,EHR000002,GT00002,1699411744
2,EHR,EHR000003,GT00003,1669205571
3,EHR,EHR000004,GT00004,1265081889
4,EHR,EHR000005,GT00005,1609044783


## 11. Remove hidden identifiers from the source datasets

`ground_truth_id` was kept temporarily while the source records and validation map were built. Remove it before saving the simulated source files.

The downstream profiling, standardization, SQL, and MDM steps should only see the source-system fields.

In [22]:
ehr_source = ehr.drop(columns="ground_truth_id").copy()
hr_source = hr.drop(columns="ground_truth_id").copy()
credentialing_source = credentialing.drop(
    columns="ground_truth_id"
).copy()


## 12. Validate the simulation

Check that source identifiers are unique, the hidden identifier is not present in operational source files, and every source record has exactly one entry in the truth map.

In [23]:
assert ehr_source["ehr_provider_id"].is_unique
assert hr_source["employee_id"].is_unique
assert credentialing_source["credentialing_id"].is_unique

assert "ground_truth_id" not in ehr_source.columns
assert "ground_truth_id" not in hr_source.columns
assert "ground_truth_id" not in credentialing_source.columns

expected_truth_rows = (
    len(ehr_source)
    + len(hr_source)
    + len(credentialing_source)
)

assert len(simulation_truth_map) == expected_truth_rows
assert simulation_truth_map["source_record_id"].notna().all()
assert simulation_truth_map["ground_truth_id"].notna().all()

print("Simulation validation checks passed.")


Simulation validation checks passed.


## 13. Review source-level data quality

Before saving, review record counts and missingness. These results document the differences intentionally introduced into each source.

In [24]:
source_summary = pd.DataFrame({
    "source": [
        "EHR",
        "HR",
        "Credentialing",
    ],
    "rows": [
        len(ehr_source),
        len(hr_source),
        len(credentialing_source),
    ],
})

source_summary


,source,rows
0,EHR,3000
1,HR,2250
2,Credentialing,2550


In [25]:
print("EHR missing values")
display(
    ehr_source
    .isna()
    .sum()
    .sort_values(ascending=False)
)

print("\nHR missing values")
display(
    hr_source
    .isna()
    .sum()
    .sort_values(ascending=False)
)

print("\nCredentialing missing values")
display(
    credentialing_source
    .isna()
    .sum()
    .sort_values(ascending=False)
)


EHR missing values


address_line_2     2667
middle_name        1468
credential          836
npi                 238
phone               156
ehr_provider_id       0
first_name            0
last_name             0
specialty_code        0
address_line_1        0
city                  0
state                 0
zip                   0
dtype: int64


HR missing values


npi                   1206
middle_initial        1161
job_credential         620
job_specialty_code     481
work_phone             279
employee_id              0
first_name               0
last_name                0
work_city                0
work_state               0
dtype: int64


Credentialing missing values


address_line_2       2201
legal_middle_name    1170
credential            708
license_number        421
license_state         371
phone                 191
npi                    51
credentialing_id        0
legal_first_name        0
legal_last_name         0
taxonomy_code           0
address_line_1          0
city                    0
state                   0
zip                     0
dtype: int64

## 14. Inspect example records

Review a small sample from each system to confirm that the schemas and formatting differences look reasonable before writing the files.

In [26]:
ehr_source.sample(
    5,
    random_state=RANDOM_SEED
)


,ehr_provider_id,npi,first_name,middle_name,last_name,credential,specialty_code,address_line_1,address_line_2,city,state,zip,phone
1801,EHR001802,1528665940,ERIK,<NA>,VILLEGAS,RN,163W00000X,2103 COSTA MESA DR,<NA>,DALLAS,TX,752282025,(469) 585-2169
1190,EHR001191,1093213555,STEPHANIE,<NA>,Dodson,"MS, RD, LD",133V00000X,2555 JIMMY JOHNSON BLVD,<NA>,PORT ARTHUR,TX,77640,(409) 853-5016
1817,EHR001818,1306723812,DANIEL,A,CABALLERO,<NA>,390200000X,5323 HARRY HINES BLVD,<NA>,DALLAS,TX,753907201,(214) 648-2168
251,EHR000252,1386119642,KELLY,NICOLE,ROSE,<NA>,225X00000X,100 MEDICAL CENTER PKWY STE 100,<NA>,HUNTSVILLE,TX,773404959,(972) 422-1860
2505,EHR002506,1841828852,BROOK,A,DANBOISE,MD,207P00000X,4502 MEDICAL DR,<NA>,SAN ANTONIO,TX,782294402,(210) 358-2078


In [27]:
hr_source.sample(
    5,
    random_state=RANDOM_SEED
)


,employee_id,npi,first_name,middle_initial,last_name,job_credential,job_specialty_code,work_city,work_state,work_phone
940,EMP000941,1588642318,NANNETTE,F,CROW,MD,208000000X,SOUTHLAKE,TX,817-680-9858
482,EMP000483,1477854107,MEGHAN,M,GREGER,"AU.D., CCC-A",231H00000X,AUSTIN,TX,5123465562
581,EMP000582,1710473475,MAHWISH,<NA>,SADRUDDIN,PA-C,363A00000X,HOUSTON,TX,713-792-6161
247,EMP000248,1700848983,LAURIE,S,HOLLEY,MD,207ZP0102X,PASADENA,TX,7137858357
1659,EMP001660,1801378310,RAFAELLE,W,TKAC,<NA>,363L00000X,WEBSTER,TX,346-617-8993


In [28]:
credentialing_source.sample(
    5,
    random_state=RANDOM_SEED
)


,credentialing_id,npi,legal_first_name,legal_middle_name,legal_last_name,credential,taxonomy_code,license_number,license_state,address_line_1,address_line_2,city,state,zip,phone
56,CRED000057,1750262325,GREGORY,NEIL,METCALF,<NA>,104100000X,<NA>,<NA>,3131 EASTSIDE ST STE 415,<NA>,HOUSTON,TX,770981919,8325367464
194,CRED000195,1598236440,OSCAR,<NA>,JIMENEZ IRAHETA,BCBA,103K00000X,<NA>,FL,12606 GREENVILLE AVE STE 260,<NA>,DALLAS,TX,752431921,3462005618
2225,CRED002226,1912689969,LOI HENRICH,S,FRANCIA,<NA>,183500000X,72569,TX,800 W OLD SETTLERS BLVD,<NA>,ROUND ROCK,TX,786812119,5122551331
233,CRED000234,1700714474,ALEXANDER,<NA>,BARDALEZ,<NA>,106S00000X,<NA>,<NA>,10777 WESTHEIMER RD STE 1100,<NA>,HOUSTON,TX,770423462,9727648880
1902,CRED001903,1811738719,AUBREY,<NA>,CLENNA,CNP,363LW0102X,1215149,TX,5200 HARRY HINES BLVD,<NA>,DALLAS,TX,752357709,2142660130


## 15. Save the simulated source files

Write the three operational source datasets separately from the hidden validation map.

The next notebook should work from the files in `data/simulated/`. The validation map should remain outside the cleaning and matching workflow until final evaluation.

In [29]:
EHR_FILE = SIMULATED_DIR / "ehr_providers.csv"
HR_FILE = SIMULATED_DIR / "hr_providers.csv"
CREDENTIALING_FILE = SIMULATED_DIR / "credentialing_providers.csv"

TRUTH_MAP_FILE = (
    VALIDATION_DIR
    / "simulation_truth_map.csv"
)

ehr_source.to_csv(
    EHR_FILE,
    index=False
)

hr_source.to_csv(
    HR_FILE,
    index=False
)

credentialing_source.to_csv(
    CREDENTIALING_FILE,
    index=False
)

simulation_truth_map.to_csv(
    TRUTH_MAP_FILE,
    index=False
)

print(f"Saved EHR source:           {EHR_FILE}")
print(f"Saved HR source:            {HR_FILE}")
print(f"Saved Credentialing source: {CREDENTIALING_FILE}")
print(f"Saved hidden truth map:     {TRUTH_MAP_FILE}")


Saved EHR source:           ../data/simulated/ehr_providers.csv
Saved HR source:            ../data/simulated/hr_providers.csv
Saved Credentialing source: ../data/simulated/credentialing_providers.csv
Saved hidden truth map:     ../data/validation/simulation_truth_map.csv


## 16. Expected outputs

After the notebook runs, the project should contain:

```text
data/
├── ground_truth/
│   └── provider_ground_truth.csv
│
├── simulated/
│   ├── ehr_providers.csv
│   ├── hr_providers.csv
│   └── credentialing_providers.csv
│
└── validation/
    └── simulation_truth_map.csv
```

The three files under `simulated/` are the inputs to the next stage: profiling, cleaning, standardization, and data-quality checks.

`simulation_truth_map.csv` remains hidden from the matching workflow and is used later to evaluate the final MDM results.